# Evaluation demo 

In [ ]:
from models.rag import rag_pipeline
from tools.rag_metrics import rouge_l,cohen_kappa,semantic_consistency
import pandas as pd
import csv
import os

c:\Users\arion\rag-classification\standard_rag\RAG_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
#  Ground Truth Dictionary 
ground_truth_map = {
    "loading_data": [
        "import pandas as pd\ndf = pd.read_csv('data.csv')",
        "import pandas as pd\nimport sqlalchemy\nengine = sqlalchemy.create_engine('sqlite:///db.sqlite')\ndf = pd.read_sql('SELECT * FROM samples', engine)",
        "import pandas as pd\ndf = pd.read_json('data.json', orient='records')"
    ],
    "preprocessing": [
        "df.fillna(0, inplace=True)\nX = df.drop('target', axis=1)\ny = df['target']",
        "from sklearn.model_selection import train_test_split\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)",
        "from sklearn.preprocessing import StandardScaler\nscaler = StandardScaler()\nX_train = scaler.fit_transform(X_train)\nX_test  = scaler.transform(X_test)",
        "from sklearn.preprocessing import LabelEncoder\nle = LabelEncoder()\ndf['category'] = le.fit_transform(df['category'])",
        "X = pd.get_dummies(df.drop('target', axis=1), drop_first=True)\ny = df['target']",
        "df['ratio'] = df['feature_a'] / (df['feature_b'] + 1e-9)\ndf['log_value'] = df['value'].apply(lambda x: x if x <= 0 else __import__('math').log(x))\ndf['interaction'] = df['feature_a'] * df['feature_b']",
        "from sklearn.decomposition import PCA\npca = PCA(n_components=0.95, random_state=42)\nX_train = pca.fit_transform(X_train)\nX_test  = pca.transform(X_test)\nprint(f'Components kept: {pca.n_components_}')",
        "from imblearn.over_sampling import SMOTE\nsm = SMOTE(random_state=42)\nX_train, y_train = sm.fit_resample(X_train, y_train)"
    ],
    "loading_model": [
        "from sklearn.ensemble import RandomForestClassifier\nmodel = RandomForestClassifier()",
        "from sklearn.svm import SVC\nmodel = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)",
        "from sklearn.ensemble import GradientBoostingClassifier\nmodel = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)",
        "from sklearn.linear_model import LogisticRegression\nmodel = LogisticRegression(max_iter=1000, random_state=42)"
    ],
    "fit": [
        "model.fit(X, y)",
        "model.fit(X_train, y_train)\ntrain_score = model.score(X_train, y_train)\ntest_score  = model.score(X_test,  y_test)\nprint(f'Train: {train_score} | Test: {test_score}')",
        "from sklearn.model_selection import cross_val_score\nscores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted')\nprint(f'CV F1: {scores.mean():.4f} ± {scores.std():.4f}')",
        "from sklearn.model_selection import GridSearchCV\nparam_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10]}\ngrid = GridSearchCV(model, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)\ngrid.fit(X_train, y_train)\nmodel = grid.best_estimator_\nprint(f'Best params: {grid.best_params_}')",
        "from sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.ensemble import RandomForestClassifier\npipeline = Pipeline([\n('scaler', StandardScaler()),\n('clf',    RandomForestClassifier(random_state=42)),\n])\npipeline.fit(X_train, y_train)"
    ],
    "results": [
        "print(f'Model Score: {model.score(X, y)}')",
        "from sklearn.metrics import classification_report\ny_pred = model.predict(X_test)\nprint(classification_report(y_test, y_pred))",
        "from sklearn.metrics import ConfusionMatrixDisplay\nimport matplotlib.pyplot as plt\ny_pred = model.predict(X_test)\nConfusionMatrixDisplay.from_predictions(y_test, y_pred)\nplt.savefig('confusion_matrix.png', bbox_inches='tight')",
        "import pandas as pd\nimportances = pd.Series(model.feature_importances_, index=X.columns)\nimportances.sort_values(ascending=False).head(10).plot(kind='bar')\nprint(importances.sort_values(ascending=False).head(10))"
    ],
    "exploratory_analysis": [
        "print(df.shape)\nprint(df.dtypes)\nprint(df.describe())\nprint(df.isnull().sum())",
        "print(y.value_counts())\nprint(y.value_counts(normalize=True).mul(100).round(2))"
    ],
    "inference": [
        "y_pred = model.predict(X_test)\ny_pred_proba = model.predict_proba(X_test)[:, 1]\nprint(y_pred[:10])",
        "import numpy as np\nsample = X_test[0].reshape(1, -1)\nlabel  = model.predict(sample)[0]\nproba  = model.predict_proba(sample).max()\nprint(f'Predicted: {label}  (confidence: {proba})"
    ]
}


In [18]:
def append_to_csv(model_name,step_name, precision,recall,f1_score,cohen_kappa,semantic_consistency, filename='rag_evaluation_results.csv'): #+ add model name
    file_exists = os.path.isfile(filename)
    with open(filename, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['model','step', 'precision', 'recall', 'f1','c_k','s_c'])
        writer.writerow([model_name,step_name, precision, recall, f1_score,cohen_kappa,semantic_consistency])

In [ ]:
#Main Evaluation Loop 
def run_evaluation(ground_truth_map):
    for step, code_chunks in ground_truth_map.items():
        for i, code_chunk in enumerate(code_chunks, 1):
        
            prediction = rag_pipeline(code_chunk) 
        
            # Calculate Metrics
            try :
                scores = rouge_l(prediction, step)
                precision,recall,f1_score = scores['precison'],scores['recall'],scores['f1']
            except :
                precision,recall,f1_score = None,None,None

            try : 
                c_k = cohen_kappa(prediction,step)
            except :
                c_k = None

            try :
                s_c = semantic_consistency(prediction,step)
            except:
                s_c = None
            # Save to CSV dynamically
            append_to_csv("RAG",step,precision,recall,f1_score,c_k,s_c) 
            print(f"Evaluated {step}")

# Execute the evaluation
run_evaluation(ground_truth_map)


NameError: name 'rag_pipeline' is not defined

In [ ]:
""" 
ground_truth_map_brut = {
 
    "loading_data": "import pandas as pd\ndf = pd.read_csv('data.csv')",
 
    "preprocessing": "df.fillna(0, inplace=True)\nX = df.drop('target', axis=1)\ny = df['target']",
 
    "loading_model": "from sklearn.ensemble import RandomForestClassifier\nmodel = RandomForestClassifier()",
 
    "fit": "model.fit(X, y)",
 
    "results": "print(f'Model Score: {model.score(X, y)}')",
 
    "loading_data": "import pandas as pd\nimport sqlalchemy\nengine = sqlalchemy.create_engine('sqlite:///db.sqlite')\ndf = pd.read_sql('SELECT * FROM samples', engine)",
 
    "loading_data": "import pandas as pd\ndf = pd.read_json('data.json', orient='records')",
 
    "exploratory_analysis": "print(df.shape)\nprint(df.dtypes)\nprint(df.describe())\nprint(df.isnull().sum())",
 
    "exploratory_analysis": "print(y.value_counts())\nprint(y.value_counts(normalize=True).mul(100).round(2))",
 
    "preprocessing": "from sklearn.model_selection import train_test_split\nX_train, X_test, y_train, y_test = train_test_split(\nX, y, test_size=0.2, random_state=42, stratify=y\n",
 
    "preprocessing": "from sklearn.preprocessing import StandardScaler\nscaler = StandardScaler()\nX_train = scaler.fit_transform(X_train)\nX_test  = scaler.transform(X_test)",
 
    "preprocessing": "from sklearn.preprocessing import LabelEncoder\nle = LabelEncoder()\ndf['category'] = le.fit_transform(df['category'])",
 
    "preprocessing": "X = pd.get_dummies(df.drop('target', axis=1), drop_first=True)\ny = df['target']",
 
    "preprocessing": "df['ratio'] = df['feature_a'] / (df['feature_b'] + 1e-9)\ndf['log_value'] = df['value'].apply(lambda x: x if x <= 0 else __import__('math').log(x))\ndf['interaction'] = df['feature_a'] * df['feature_b']",
 
    "preprocessing": "from sklearn.decomposition import PCA\npca = PCA(n_components=0.95, random_state=42)\nX_train = pca.fit_transform(X_train)\nX_test  = pca.transform(X_test)\nprint(f'Components kept: {pca.n_components_}')",
 
    "preprocessing": "from imblearn.over_sampling import SMOTE\nsm = SMOTE(random_state=42)\nX_train, y_train = sm.fit_resample(X_train, y_train)",
 
    "loading_model": "from sklearn.svm import SVC\nmodel = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)",
 
    "loading_model": "from sklearn.ensemble import GradientBoostingClassifier\nmodel = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)",
 
    "loading_model": "from sklearn.linear_model import LogisticRegression\nmodel = LogisticRegression(max_iter=1000, random_state=42)",
 
    "fit": "model.fit(X_train, y_train)\ntrain_score = model.score(X_train, y_train)\ntest_score  = model.score(X_test,  y_test)\nprint(f'Train: {train_score} | Test: {test_score}')",
 
    "fit": "from sklearn.model_selection import cross_val_score\nscores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted')\nprint(f'CV F1: {scores.mean():.4f} ± {scores.std():.4f}')",
 
    "fit": "from sklearn.model_selection import GridSearchCV\nparam_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10]}\ngrid = GridSearchCV(model, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)\ngrid.fit(X_train, y_train)\nmodel = grid.best_estimator_\nprint(f'Best params: {grid.best_params_}')",
 
    "fit": "from sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.ensemble import RandomForestClassifier\npipeline = Pipeline([\n('scaler', StandardScaler()),\n('clf',    RandomForestClassifier(random_state=42)),\n])\npipeline.fit(X_train, y_train)",
 
    "results": "from sklearn.metrics import classification_report\ny_pred = model.predict(X_test)\nprint(classification_report(y_test, y_pred))",
 
    "results": "from sklearn.metrics import ConfusionMatrixDisplay\nimport matplotlib.pyplot as plt\ny_pred = model.predict(X_test)\nConfusionMatrixDisplay.from_predictions(y_test, y_pred)\nplt.savefig('confusion_matrix.png', bbox_inches='tight')",
 
    "results": "import pandas as pd\nimportances = pd.Series(model.feature_importances_, index=X.columns)\nimportances.sort_values(ascending=False).head(10).plot(kind='bar')\nprint(importances.sort_values(ascending=False).head(10))",
 
    "inference": "y_pred       = model.predict(X_test)\ny_pred_proba = model.predict_proba(X_test)[:, 1]\nprint(y_pred[:10])",
 
    "inference": "import numpy as np\nsample = X_test[0].reshape(1, -1)\nlabel  = model.predict(sample)[0]\nproba  = model.predict_proba(sample).max()\nprint(f'Predicted: {label}  (confidence: {proba})",
 
}
 
# ---------------------------------------------------------------------------
# Sanity check
# ---------------------------------------------------------------------------
print(len(ground_truth_map))
print(f"Dataset size: {len(ground_truth_map)} examples")
print("Labels:", list(ground_truth_map.keys()))
"""

7
Dataset size: 7 examples
Labels: ['loading_data', 'preprocessing', 'loading_model', 'fit', 'results', 'exploratory_analysis', 'inference']


Step: loading_data
  Code Chunk 1:
  import pandas as pd
df = pd.read_csv('data.csv')

  Code Chunk 2:
  import pandas as pd
import sqlalchemy
engine = sqlalchemy.create_engine('sqlite:///db.sqlite')
df = pd.read_sql('SELECT * FROM samples', engine)

  Code Chunk 3:
  import pandas as pd
df = pd.read_json('data.json', orient='records')

Step: preprocessing
  Code Chunk 1:
  df.fillna(0, inplace=True)
X = df.drop('target', axis=1)
y = df['target']

  Code Chunk 2:
  from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

  Code Chunk 3:
  from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

  Code Chunk 4:
  from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['category'] = le.fit_transform(df['category'])

  Code Chunk 5:
  X = pd.get_dummies(df.drop('target', axis=1), drop_